# 01. Pothole / Manhole Detection with YOLOv8

AI-Hub ?? ??? annotation? YOLO ???? ????, ?????? ?? ??? ??/?????. ?? ??? ???, ??, ??, ?? ??? ??, rule-based damage summary ??? ???? ???????.

## 1. Colab setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/pm-road-risk-map')
if PROJECT_DIR.exists():
    sys.path.append(str(PROJECT_DIR / 'src'))

from road_risk.pothole_manhole import (
    convert_aihub_json_to_yolo,
    write_split_files,
    summarize_yolo_predictions,
)
from road_risk.risk_scoring import calculate_damage_score, grade_damage

## 2. Paths

Google Drive ??? ?? ?? ?????. ?? Drive ??? ??? ??? ???? ???.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
RAW_JSON_DIR = DRIVE_ROOT / 'mpdata/raw_annotations'
RAW_IMAGE_DIR = DRIVE_ROOT / 'mpdata/raw_images'
TRAIN_DIR = DRIVE_ROOT / 'mpdata/train'
LABEL_DIR = TRAIN_DIR / 'labels'
IMAGE_DIR = TRAIN_DIR / 'images'
YAML_PATH = TRAIN_DIR / 'data_noneg.yaml'

SEOUL_IMAGE_DIR = DRIVE_ROOT / 'seoul_images/matched_images'
SEOUL_PRED_DIR = DRIVE_ROOT / 'seoul_images/road_damage_predictions'
SEOUL_SUMMARY_CSV = DRIVE_ROOT / 'seoul_images/road_damage_summary.csv'

for p in [TRAIN_DIR, LABEL_DIR, SEOUL_PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

## 3. Convert labels to YOLO format

???? ???? `category_id=8`? ???, `category_id=10`? ?????.

In [ ]:
summary = convert_aihub_json_to_yolo(RAW_JSON_DIR, LABEL_DIR)
summary.head()

In [ ]:
summary.groupby('class_id')['bbox_area_ratio'].describe()

## 4. Train / validation / test split

In [ ]:
split_paths = write_split_files(IMAGE_DIR, TRAIN_DIR, val_size=0.1, test_size=0.1, seed=42)
split_paths

In [ ]:
yaml_text = f"""
path: {TRAIN_DIR}
train: train.txt
val: val.txt
test: test.txt
names:
  0: pothole
  1: manhole
""".strip()
YAML_PATH.write_text(yaml_text, encoding='utf-8')
print(YAML_PATH.read_text())

## 5. Train YOLOv8

??????? ?? ???? ??? ?? ? pothole mAP50? 0.0939?? 0.476?? ???????.

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n.pt')

model.train(
    data=str(YAML_PATH),
    epochs=100,
    imgsz=640,
    batch=8,
    patience=20,
    device=device,
    project=str(DRIVE_ROOT / 'yolo_runs'),
    name='road_damage_yolov8n_640',
)

## 6. Evaluate best model

In [ ]:
best_model_path = DRIVE_ROOT / 'yolo_runs/road_damage_yolov8n_640/weights/best.pt'
model = YOLO(str(best_model_path))
val_metrics = model.val(data=str(YAML_PATH), imgsz=640, split='val')
test_metrics = model.val(data=str(YAML_PATH), imgsz=640, split='test')
print(val_metrics)
print(test_metrics)

## 7. Predict Seoul road images

In [ ]:
model = YOLO(str(best_model_path))
results = model.predict(
    source=str(SEOUL_IMAGE_DIR),
    imgsz=640,
    conf=0.25,
    save=True,
    save_txt=True,
    project=str(SEOUL_PRED_DIR.parent),
    name=SEOUL_PRED_DIR.name,
)

## 8. Build image-level damage summary

In [ ]:
pred_label_dir = SEOUL_PRED_DIR / 'labels'
damage_df = summarize_yolo_predictions(pred_label_dir, SEOUL_IMAGE_DIR)
damage_df['damage_score'] = damage_df.apply(
    lambda r: calculate_damage_score(
        r['pothole_count'],
        r['manhole_count'],
        r['pothole_area_ratio'],
        r['max_pothole_area_ratio'],
    ),
    axis=1,
)
damage_df['damage_grade'] = damage_df['damage_score'].map(grade_damage)
damage_df.to_csv(SEOUL_SUMMARY_CSV, index=False, encoding='utf-8-sig')
damage_df.head()

In [ ]:
damage_df['damage_grade'].value_counts().sort_index()